In [ ]:
import os
import json
import re
import glob
import hashlib
import argparse
from pathlib import Path
from typing import Any
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from tqdm import tqdm

load_dotenv()


groq_api = os.getenv("GROQ_API_KEY")
if groq_api is None:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------


SCHEME_DIR = Path("data/schemes")
FAQ_FILE = Path("data/faqs.csv")
OUTPUT_DIR = Path("output")

OUTPUT_DIR.mkdir(exist_ok=True)

QUESTIONS_PER_SCHEME = 5

llm = Groq(api_key=groq_api)


In [6]:
# --------------------------------------------------
# 2. LOAD SCHEME JSON FILES
# --------------------------------------------------

def load_schemes(folder: Path) -> list[dict]:

    schemes = []

    for file_path in glob.glob(str(folder / "*.json")):

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Support either one scheme or a list of schemes
            if isinstance(data, list):
                schemes.extend(data)

            elif isinstance(data, dict):
                schemes.append(data)

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    print(f"Loaded {len(schemes)} schemes")

    return schemes

In [7]:
# --------------------------------------------------
# 3. LOAD FAQ CSV
# --------------------------------------------------

def load_faqs(file_path: Path) -> pd.DataFrame:

    df = pd.read_csv(file_path)

    required_columns = {
        "scheme_id",
        "name",
        "question",
        "answer"
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing FAQ CSV columns: {missing}")

    df["scheme_id"] = df["scheme_id"].astype(str).str.strip()

    df = df.fillna("")

    print(f"Loaded {len(df)} FAQs")

    return df



In [8]:
# --------------------------------------------------
# 4. NORMALIZE TEXT FOR DUPLICATE DETECTION
# --------------------------------------------------

def normalize_text(text: str) -> str:

    text = str(text).lower().strip()

    text = re.sub(r"[^\w\s]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
# --------------------------------------------------
# 5. CREATE ORIGINAL FAQ RECORDS
# --------------------------------------------------

def create_faq_records(scheme: dict, faq_rows: pd.DataFrame) -> list[dict]:

    records = []

    scheme_id = str(scheme.get("scheme_id", ""))

    for index, faq in faq_rows.iterrows():

        question = str(faq["question"]).strip()
        answer = str(faq["answer"]).strip()

        if not question or not answer:
            continue

        faq_id = f"{scheme_id}_FAQ_{index + 1}"

        records.append({
            "scheme_id": scheme_id,
            "scheme_name": scheme.get("name", ""),

            "question": question,
            "expected_answer": answer,

            "expected_facts": [],

            "relevant_doc_ids": [
                f"{scheme_id}_FAQ"
            ],

            "evidence": [{
                "source_field": "original_faq_answer",
                "quote": answer
            }],

            "question_type": "faq",
            "question_source": "original_faq",

            "difficulty": "easy",
            "language": "en",

            "answerable": True,
            "verification_status": "pending",

            "source_faq_id": faq_id,
            "source_url": faq.get("url", "")
        })

    return records


In [10]:
# --------------------------------------------------
# 6. PREPARE SCHEME INFORMATION FOR THE LLM
# --------------------------------------------------

def prepare_scheme_context(scheme: dict) -> dict:

    fields = [
        "scheme_id",
        "name",
        "level",
        "ministry",
        "nodal_department",
        "states",
        "categories",
        "target_beneficiaries",
        "status",
        "brief_description",
        "benefits",
        "eligibility",
        "exclusions",
        "documents_required",
        "application_process"
    ]

    context = {}

    for field in fields:

        value = scheme.get(field)

        if value not in [None, "", [], {}]:
            context[field] = value

    return context


In [ ]:
# --------------------------------------------------
# 7. GENERATE QUESTIONS USING GROQ
# --------------------------------------------------

def generate_questions(
    scheme: dict,
    existing_faqs: pd.DataFrame,
    number_of_questions: int = QUESTIONS_PER_SCHEME
) -> list[dict]:

    scheme_context = prepare_scheme_context(scheme)

    existing_questions = (
        existing_faqs["question"].tolist()
        if not existing_faqs.empty
        else []
    )

    prompt = f"""
You are an expert dataset curator for a government scheme
discovery and eligibility assistant called SEVA AI.

Your task is to generate evaluation questions using ONLY
the supplied scheme information.

SCHEME INFORMATION:
{json.dumps(scheme_context, ensure_ascii=False, indent=2)}

EXISTING FAQ QUESTIONS:
{json.dumps(existing_questions, ensure_ascii=False, indent=2)}

Generate exactly {number_of_questions} NEW questions.

REQUIREMENTS:

1. Do not repeat existing FAQ questions.
2. Do not invent scheme benefits or eligibility rules.
3. Every expected answer must be supported by the supplied data.
4. Generate realistic questions that citizens may ask.
5. Cover different topics where the source information allows:
   - eligibility
   - benefits
   - required documents
   - application process
   - exclusions
   - scheme discovery
6. Include simple and moderately difficult questions.
7. Avoid multiple questions testing exactly the same fact.
8. If information is missing, do not invent it.
9. Use the same language as the source information.
10. Each evidence quote must be copied from the supplied
    scheme information as accurately as possible.

Return ONLY valid JSON in this format:

{{
  "questions": [
    {{
      "question": "Question asked by a citizen",
      "expected_answer": "Answer supported by the source",
      "expected_facts": [
        "Fact 1",
        "Fact 2"
      ],
      "evidence": [
        {{
          "source_field": "eligibility",
          "quote": "Supporting source text"
        }}
      ],
      "question_type": "eligibility",
      "difficulty": "easy",
      "language": "en"
    }}
  ]
}}

Allowed question_type values:
eligibility, benefits, documents, application,
exclusions, discovery, status, other.

Allowed difficulty values:
easy, medium, hard.

Do not add Markdown fences or explanations.
"""

    try:

        response = llm.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You generate source-grounded evaluation "
                        "datasets. Never invent facts."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.1,
            max_tokens=5000
        )

        content = response.choices[0].message.content.strip()

        # Handle accidental Markdown code fences
        content = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            content
        )

        result = json.loads(content)

        return result.get("questions", [])

    except Exception as e:

        print(
            f"LLM generation failed for "
            f"{scheme.get('name')}: {e}"
        )

        return []



In [12]:
# --------------------------------------------------
# 8. CONVERT GENERATED QUESTIONS INTO RECORDS
# --------------------------------------------------

def create_generated_records(
    scheme: dict,
    questions: list[dict]
) -> list[dict]:

    records = []

    scheme_id = str(scheme.get("scheme_id", ""))
    scheme_name = scheme.get("name", "")

    for index, q in enumerate(questions, start=1):

        question = str(q.get("question", "")).strip()
        answer = str(q.get("expected_answer", "")).strip()

        if not question or not answer:
            continue

        records.append({

            "scheme_id": scheme_id,
            "scheme_name": scheme_name,

            "question": question,
            "expected_answer": answer,

            "expected_facts": q.get("expected_facts", []),

            "relevant_doc_ids": [
                f"{scheme_id}_{q.get('question_type', 'other')}"
            ],

            "evidence": q.get("evidence", []),

            "question_type": q.get("question_type", "other"),
            "question_source": "llm_generated",

            "difficulty": q.get("difficulty", "medium"),
            "language": q.get("language", "en"),

            "answerable": True,
            "verification_status": "pending",

            "source_faq_id": None,
            "source_url": scheme.get("myscheme_url", "")
        })

    return records


In [13]:
# --------------------------------------------------
# 9. REMOVE EXACT DUPLICATE QUESTIONS
# --------------------------------------------------

def remove_duplicates(records: list[dict]) -> list[dict]:

    unique_records = []
    seen = set()

    for record in records:

        normalized = normalize_text(record["question"])

        # Include scheme ID so questions for different schemes
        # are not incorrectly merged.
        key = (
            record["scheme_id"],
            normalized
        )

        if key in seen:
            continue

        seen.add(key)
        unique_records.append(record)

    print(
        f"Removed {len(records) - len(unique_records)} "
        "exact duplicate questions"
    )

    return unique_records


In [14]:
# --------------------------------------------------
# 10. ASSIGN UNIQUE TEST IDS
# --------------------------------------------------

def assign_test_ids(records: list[dict]) -> list[dict]:

    for index, record in enumerate(records, start=1):

        record["test_id"] = f"GD_{index:06d}"

    return records


# --------------------------------------------------
# 11. SAVE OUTPUT FILES
# --------------------------------------------------

def save_outputs(records: list[dict]):

    json_path = OUTPUT_DIR / "golden_dataset_candidates.json"
    jsonl_path = OUTPUT_DIR / "golden_dataset_candidates.jsonl"
    csv_path = OUTPUT_DIR / "golden_dataset_candidates.csv"

    # Full JSON
    with open(json_path, "w", encoding="utf-8") as f:

        json.dump(
            records,
            f,
            ensure_ascii=False,
            indent=2
        )

    # JSONL: one record per line
    with open(jsonl_path, "w", encoding="utf-8") as f:

        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # CSV: nested fields stored as JSON strings
    df = pd.DataFrame(records)

    for column in [
        "expected_facts",
        "relevant_doc_ids",
        "evidence"
    ]:

        df[column] = df[column].apply(
            lambda value: json.dumps(
                value,
                ensure_ascii=False
            )
        )

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print("\nOutput files created:")
    print(json_path)
    print(jsonl_path)
    print(csv_path)


In [ ]:
# --------------------------------------------------
# 12. MAIN PIPELINE
# --------------------------------------------------

def main(limit: int | None = None):

    schemes = load_schemes(SCHEME_DIR)
    faqs = load_faqs(FAQ_FILE)

    if limit:
        schemes = schemes[:limit]

    all_records = []

    for scheme in tqdm(schemes, desc="Processing schemes"):

        scheme_id = str(scheme.get("scheme_id", "")).strip()

        if not scheme_id:
            continue

        scheme_faqs = faqs[
            faqs["scheme_id"] == scheme_id
        ]

        # A. Preserve existing FAQ questions
        faq_records = create_faq_records(
            scheme,
            scheme_faqs
        )

        all_records.extend(faq_records)

        # B. Generate additional questions
        generated_questions = generate_questions(
            scheme,
            scheme_faqs
        )

        generated_records = create_generated_records(
            scheme,
            generated_questions
        )

        all_records.extend(generated_records)

    # C. Deduplicate
    all_records = remove_duplicates(all_records)

    # D. Assign IDs
    all_records = assign_test_ids(all_records)

    # E. Export
    save_outputs(all_records)

    print("\n========== SUMMARY ==========")

    print("Schemes processed:", len(schemes))
    print("Total candidate questions:", len(all_records))

    print(
        "Original FAQ questions:",
        sum(
            r["question_source"] == "original_faq"
            for r in all_records
        )
    )

    print(
        "LLM-generated questions:",
        sum(
            r["question_source"] == "llm_generated"
            for r in all_records
        )
    )


if __name__ == "__main__":

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help="Process only the first N schemes for testing"
    )

    args = parser.parse_args()

    main(limit=args.limit)